In [ ]:
# ==============================================================================
# REQUIRED MODULE IMPORTS
# ==============================================================================
import io                               # Allows reading text strings directly as file streams in memory (no disk saving required)
import csv                              # Python's built-in CSV parser (handles quote-heavy & JSON columns without crashing)
import time                             # Provides delay functionality for retry backoffs when encountering server timeouts
import requests                         # Handles HTTP network calls to communicate with and download files from the FAA server
import pandas as pd                     # Data manipulation library used to stitch, clean, deduplicate, and join datasets
from concurrent.futures import ThreadPoolExecutor, as_completed # Enables multi-threading for fast, parallel downloading

# ==============================================================================
# 1. GLOBAL CONFIGURATION & SCRAPING SETTINGS
# ==============================================================================
# Define all 11 FAA regional codes to target
REGIONS = ['AAL', 'ACE', 'AEA', 'AGL', 'ANE', 'ANM', 'ASO', 'ASW', 'AWP', 'WTE', 'WTW']

# Define the full historical year coverage scope (1960 through 2026)
START_YEAR = 1960
END_YEAR = 2026

# Target base API URL endpoint for OE/AAA file archives
BASE_URL = "https://oeaaa.faa.gov/oeaaa/oe3a-external-api/downloadArchives.do"

# List of specific column header names we want to extract from every downloaded CSV file
TARGET_COLS = ['STUDY (ASN)', 'ENTERED DATE', 'FCC NUMBER']

# HTTP headers to mimic a genuine web browser session and avoid connection blocks by the server
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': '*/*',
    'Referer': 'https://oeaaa.faa.gov/oeaaa/oe3a/main/'
}

# Define local input file name (your database reference) and final output file name
INPUT_FILE = "Result 2026-08-04 01-24-03.csv"
OUTPUT_FILE = "Updated_Result_2026-08-04.csv"

# ==============================================================================
# 2. LOAD EXISTING REFERENCE DATA (INPUT CSV)
# ==============================================================================
# Load your database reference CSV file into a Pandas DataFrame
try:
    ref_df = pd.read_csv(INPUT_FILE)
    # Print loading confirmation immediately to screen (flush=True forces Google Colab to display output without buffering)
    print(f"📁 Successfully loaded '{INPUT_FILE}' ({len(ref_df):,} records).", flush=True)
except FileNotFoundError:
    # Gracefully handle missing file scenarios without breaking the script execution
    print(f"⚠️ Warning: Local file '{INPUT_FILE}' not found. Please upload to Google Colab.", flush=True)
    ref_df = pd.DataFrame()

# ==============================================================================
# 3. STREAM READER WORKER FUNCTION (FETCH & EXTRACT)
# ==============================================================================
def fetch_region_year(region, year):
    """
    Downloads a single region-year archive CSV from the FAA server into memory,
    parses rows using a robust string-stream reader to safely bypass quote/JSON errors,
    and extracts only the three required target columns.
    """
    # Construct filename and full download URL according to FAA naming convention
    fname = f"Part77{region}{year}List.gzip"
    url = f"{BASE_URL}?fname={fname}"

    # Attempt network request up to 2 times in case of temporary network glitches
    for attempt in range(2):
        try:
            # Send HTTP GET request with an 8-second timeout window
            response = requests.get(url, headers=HEADERS, timeout=8)

            # Validate that server returned HTTP 200 (Success) and non-empty content
            if response.status_code == 200 and len(response.content) > 100:
                # Convert raw text response into an in-memory string stream (io.StringIO)
                f = io.StringIO(response.text)

                # Use standard csv.reader to navigate line by line safely
                reader = csv.reader(f)

                # Extract header row and strip any leading/trailing whitespace from column names
                header = [col.strip() for col in next(reader, [])]

                # Map column header names to their numerical column index positions (0-based)
                col_indices = {name: header.index(name) for name in TARGET_COLS if name in header}

                # If required target columns are missing in this file, log and exit worker
                if not col_indices:
                    return (False, region, year, None, f"⚠️ Missing columns: {region}-{year}")

                # Determine the maximum column index we need to inspect per row
                max_idx = max(col_indices.values())
                extracted_data = []

                # Iterate through all data rows in the file
                for row in reader:
                    # Ensure the current row has enough columns to avoid IndexError
                    if len(row) > max_idx:
                        # Extract only the 3 required fields and trim whitespace
                        extracted_data.append({
                            name: row[idx].strip() for name, idx in col_indices.items()
                        })

                # If valid rows were extracted, wrap in DataFrame and return success
                if extracted_data:
                    df_temp = pd.DataFrame(extracted_data)
                    return (True, region, year, df_temp, f"✅ Loaded {region}-{year}: {len(df_temp):,} records")
                else:
                    return (False, region, year, None, f"⚪ Empty file: {region}-{year}")

            # Handle 404 response (indicates no data archive exists for this specific region-year)
            elif response.status_code == 404:
                return (False, region, year, None, f"⚪ 404 Not Found: {region}-{year}")

        except Exception:
            # On first failed attempt, pause briefly and retry; on second failure, register timeout
            if attempt == 1:
                return (False, region, year, None, f"⚠️ Timeout: {region}-{year}")
            time.sleep(0.5)

    # Default fallback return when no data could be retrieved
    return (False, region, year, None, f"⚪ No data: {region}-{year}")

# ==============================================================================
# 4. MULTI-THREADED BULK SCRAPING EXECUTION
# ==============================================================================
# Display initial start message (Requirement #11 Logging)
print(f"\n🚀 Extracting files from 1960 to 2026 across all 11 regions…\n", flush=True)

# Generate list of years in reverse chronological order (2026 down to 1960) to pull newest data first
years = list(range(END_YEAR, START_YEAR - 1, -1))

# Create a list of all 737 region-year tuples to process (11 regions x 67 years)
tasks = [(region, year) for year in years for region in REGIONS]

# Storage containers for successful DataFrames, file counters, and error logs
all_dfs = []
successful_files_count = 0
failed_logs = []

# Set parallel execution thread pool size (downloads up to 12 files concurrently)
MAX_WORKERS = 12

# Execute tasks asynchronously across parallel threads
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Submit all fetching tasks to the thread pool executor
    futures = [executor.submit(fetch_region_year, region, year) for region, year in tasks]

    # Process each task as it completes (in order of thread completion, not task list order)
    for future in as_completed(futures):
        success, region, year, df_temp, message = future.result()

        if success and df_temp is not None:
            # Collect DataFrame and increment counter
            all_dfs.append(df_temp)
            successful_files_count += 1
            # Output live completion status (Requirement #12 Logging)
            print(message, flush=True)
        else:
            # Store failure message in log collection (Requirement #15 Logging collection)
            failed_logs.append(message)

# ==============================================================================
# 5. DATABASE STITCHING & CONSOLIDATION
# ==============================================================================
# Display extraction summary count (Requirement #13 Logging)
print(f"\nSuccessfully extracted {successful_files_count} files! Stitching database…", flush=True)

# Combine all individually extracted DataFrames into one large master DataFrame
if all_dfs:
    master_df = pd.concat(all_dfs, ignore_index=True)
    # Output total row count combined (Requirement #14 Logging)
    print(f"📊 Total records stitched across all files: {len(master_df):,}", flush=True)
else:
    master_df = pd.DataFrame(columns=TARGET_COLS)
    print("⚠️ No records were extracted!", flush=True)

# Print log summary of unextracted/missing files (Requirement #15 Logging output)
if failed_logs:
    print(f"\n📋 Unextracted / Missing files logged ({len(failed_logs)} total):")
    # Show first 15 entries as a clean summary preview
    for log in failed_logs[:15]:
        print(f"   • {log}")
    if len(failed_logs) > 15:
        print(f"   ... and {len(failed_logs) - 15} more.")

# ==============================================================================
# 6. DEDUPLICATION (RETAIN LATEST ENTERED DATE PER FCC NUMBER)
# ==============================================================================
if not master_df.empty and 'FCC NUMBER' in master_df.columns and 'ENTERED DATE' in master_df.columns:
    print("\n🧹 Sorting and deduplicating to keep only the latest ENTERED DATE per FCC NUMBER...", flush=True)

    # Standardize string formatting for clean matching
    master_df['FCC NUMBER'] = master_df['FCC NUMBER'].astype(str).str.strip()

    # Convert string dates into proper Datetime objects for chronological sorting
    master_df['ENTERED DATE_DT'] = pd.to_datetime(master_df['ENTERED DATE'], errors='coerce')

    # Filter out empty, missing, or invalid FCC Numbers
    valid_fcc = master_df[
        (master_df['FCC NUMBER'].notna()) &
        (master_df['FCC NUMBER'] != '') &
        (master_df['FCC NUMBER'] != 'nan')
    ].copy()

    # Sort dataset: FCC NUMBER ascending, then ENTERED DATE descending (newest date on top)
    valid_fcc = valid_fcc.sort_values(by=['FCC NUMBER', 'ENTERED DATE_DT'], ascending=[True, False])

    # Drop duplicate FCC NUMBER rows, keeping the first occurrence (which is the newest date)
    curated_fcc_df = valid_fcc.drop_duplicates(subset=['FCC NUMBER'], keep='first').drop(columns=['ENTERED DATE_DT'])

    print(f"🎯 Retained {len(curated_fcc_df):,} unique FCC NUMBER records with their latest information.", flush=True)
else:
    curated_fcc_df = pd.DataFrame(columns=TARGET_COLS)

# ==============================================================================
# 7. LEFT JOIN ON REFERENCE FILE & EXPORT FINAL CSV
# ==============================================================================
if not ref_df.empty and not curated_fcc_df.empty:
    print("\n🔗 Executing Left Join: Reference dataset ON fcc_asr_number = FCC NUMBER...", flush=True)

    # Step 1: Create a temporary normalized join key column on the reference table
    ref_df['fcc_asr_key'] = ref_df['fcc_asr_number'].astype(str).str.strip()

    # Step 2: Remove any overlapping column names from ref_df to prevent Pandas from making _x and _y suffixes
    overlapping_cols = [col for col in curated_fcc_df.columns if col in ref_df.columns]
    ref_df_clean = ref_df.drop(columns=overlapping_cols, errors='ignore')

    # Step 3: Perform Left Join keeping all reference rows intact while appending latest FAA records
    merged_df = ref_df_clean.merge(
        curated_fcc_df,
        left_on='fcc_asr_key',
        right_on='FCC NUMBER',
        how='left'
    ).drop(columns=['fcc_asr_key']) # Clean up temporary key column

    # Step 4: Export final merged DataFrame to CSV
    merged_df.to_csv(OUTPUT_FILE, index=False)
    print(f"💾 Complete! Saved final updated dataset to '{OUTPUT_FILE}' ({len(merged_df):,} total rows).", flush=True)

📁 Successfully loaded 'Result 2026-08-04 01-24-03.csv' (166,542 records).

🚀 Extracting files from 1960 to 2026 across all 11 regions…

✅ Loaded AAL-2025: 1,230 records
✅ Loaded AAL-2026: 2,241 records
✅ Loaded WTE-2026: 3,601 records
✅ Loaded WTW-2026: 7,433 records
✅ Loaded ANE-2026: 2,644 records
✅ Loaded ANM-2026: 5,766 records
✅ Loaded ACE-2026: 5,995 records
✅ Loaded AEA-2026: 10,612 records
✅ Loaded ANE-2025: 4,344 records
✅ Loaded AWP-2026: 13,179 records
✅ Loaded ASW-2026: 14,226 records
✅ Loaded ACE-2025: 9,506 records
✅ Loaded AAL-2024: 1,762 records
✅ Loaded ANM-2025: 8,546 records
✅ Loaded ASO-2026: 17,963 records
✅ Loaded WTE-2025: 7,875 records
✅ Loaded AGL-2026: 21,147 records
✅ Loaded WTW-2025: 14,356 records
✅ Loaded ACE-2024: 7,350 records
✅ Loaded AEA-2025: 14,980 records
✅ Loaded ANE-2024: 5,385 records
✅ Loaded ASW-2025: 17,865 records
✅ Loaded AWP-2025: 18,023 records
✅ Loaded AGL-2025: 22,660 records
✅ Loaded WTE-2024: 7,820 records
✅ Loaded WTW-2024: 12,840 rec

In [ ]:
merged_df

,fcc_asr_number,faa_study_number,STUDY (ASN),ENTERED DATE,FCC NUMBER
0,1000002,2001-ASO-2903-OE,2022-ASO-36287-OE,2022-09-21,1000002
1,1000003,1996-ASO-2814-OE,2022-ASO-2252-OE,2022-01-18,1000003
2,1000004,2006-ASO-6080-OE,2021-ASO-46712-OE,2021-11-20,1000004
3,1000007,NaN,2018-ASW-9067-OE,2018-06-12,1000007
4,1000008,2009-AGL-1012-OE,2009-AGL-1012-OE,2009-03-03,1000008
...,...,...,...,...,...
166537,1330865,NaN,2025-ASO-4966-OE,2025-03-10,1330865
166538,1330947,2024-ACE-6738-OE,NaN,NaN,NaN
166539,1331072,2024-ASO-9579-OE,2024-ASO-9579-OE,2024-05-06,1331072
166540,1331683,NaN,2025-AGL-10102-OE,2025-07-07,1331683


In [ ]:
merged_df[merged_df['faa_study_number'] != merged_df['STUDY (ASN)']] # fcc_asr_number & faa_study_number are from the database while the rest are from the scraped information.

,fcc_asr_number,faa_study_number,STUDY (ASN),ENTERED DATE,FCC NUMBER
0,1000002,2001-ASO-2903-OE,2022-ASO-36287-OE,2022-09-21,1000002
1,1000003,1996-ASO-2814-OE,2022-ASO-2252-OE,2022-01-18,1000003
2,1000004,2006-ASO-6080-OE,2021-ASO-46712-OE,2021-11-20,1000004
3,1000007,NaN,2018-ASW-9067-OE,2018-06-12,1000007
5,1000009,1972-CE-424-OE,NaN,NaN,NaN
...,...,...,...,...,...
166533,1330328,2024-AEA-6235-OE,2026-AEA-9390-OE,2026-08-05,1330328
166537,1330865,NaN,2025-ASO-4966-OE,2025-03-10,1330865
166538,1330947,2024-ACE-6738-OE,NaN,NaN,NaN
166540,1331683,NaN,2025-AGL-10102-OE,2025-07-07,1331683
